# OpenAtlas — N-ATLaS Hosted Endpoint (Colab)

Serves the real **N-ATLaS LLM** (`NCAIR1/N-ATLaS`, Llama-3 8B, 4-bit) plus all **four N-ATLaS ASR models**
(`NCAIR1/Yoruba-ASR`, `NCAIR1/Hausa-ASR`, `NCAIR1/Igbo-ASR`, `NCAIR1/NigerianAccentedEnglish` — all Whisper-small
fine-tunes) behind one FastAPI server, tunneled to a public HTTPS URL.

This is the thing the OpenAtlas SDK's `chat()` and `transcribe()` methods call. It is **not** persistent —
free Colab kills the session on idle timeout or after ~12h, and the tunnel URL changes every run. Treat this
as the "keep it warm for the demo / submission window" deployment, not production hosting (see the
Architecture doc's RunPod Serverless plan for the real persistent version — this notebook is the fast path
to a *working, genuine* N-ATLaS endpoint today).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Request access to `NCAIR1/N-ATLaS` on Hugging Face (one-click "Agree and access repository" — approval is usually instant).
3. Colab secrets (key icon, left sidebar) → add:
   - `HF_TOKEN` — a Hugging Face **read** token, notebook access ON
   - `NATLAS_API_KEY` — any long random string you make up; this becomes the API key OpenAtlas developers send
4. Runtime → Run all.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn[standard] \
    python-multipart nest-asyncio pyngrok soundfile librosa
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!ffmpeg -version | head -1  # audio decoding (webm, m4a, mp3, ...) goes through ffmpeg


## 2. Load secrets

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
NATLAS_API_KEY = userdata.get('NATLAS_API_KEY')
assert HF_TOKEN, "Add HF_TOKEN in Colab secrets (key icon, left sidebar)"
assert NATLAS_API_KEY, "Add NATLAS_API_KEY in Colab secrets — any long random string"

os.environ["HF_TOKEN"] = HF_TOKEN
print("Secrets loaded.")


## 3. Load the N-ATLaS LLM (4-bit, T4)

This is the actual reasoning model — the only thing allowed to answer in `chat()`. No fallback to a
general-purpose model lives in this notebook, on purpose: that's the thing NAIC disqualifies submissions for.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_REPO = "NCAIR1/N-ATLaS"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading N-ATLaS LLM (this takes a few minutes on first run)...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_REPO, token=HF_TOKEN)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_REPO,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
llm_model.eval()
print("N-ATLaS LLM loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 4. Load the four N-ATLaS ASR models (Whisper-small fine-tunes)

In [ ]:
from transformers import pipeline

ASR_REPOS = {
    "yo":    "NCAIR1/Yoruba-ASR",
    "ha":    "NCAIR1/Hausa-ASR",
    "ig":    "NCAIR1/Igbo-ASR",
    "en-ng": "NCAIR1/NigerianAccentedEnglish",
}

asr_pipelines = {}
for lang_code, repo in ASR_REPOS.items():
    print(f"Loading {repo} ({lang_code})...")
    asr_pipelines[lang_code] = pipeline(
        "automatic-speech-recognition",
        model=repo,
        token=HF_TOKEN,
        device=0 if torch.cuda.is_available() else -1,
    )

print("All 4 ASR models loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. FastAPI server

Mirrors the API surface in `OpenAtlas_Architecture_DevPlan.md` section 3, so the SDK's `chat()` and
`transcribe()` methods need zero changes to point at this instead of the final RunPod deployment later —
only `OPENATLAS_BASE_URL` changes.

In [ ]:
import os, subprocess, tempfile, threading, time
from fastapi import FastAPI, Header, HTTPException, UploadFile, File, Form
from pydantic import BaseModel
from typing import List, Optional
import numpy as np

app = FastAPI(title="OpenAtlas N-ATLaS Endpoint")

# One GPU job at a time. Requests run in parallel threads; two generate()/ASR calls at once on a
# 16 GB T4 can run out of memory, so they queue here instead.
gpu_lock = threading.Lock()

SAMPLE_RATE = 16000   # what the Whisper models use
PIECE_SECONDS = 25    # long audio is cut into plain pieces of this length (see deploy/REPORT.md, KI-11)

def decode_audio(raw: bytes) -> np.ndarray:
    """Any format ffmpeg reads (wav, mp3, ogg, flac, webm, m4a) -> 16 kHz mono float32."""
    with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as f:
        f.write(raw)
        path = f.name
    try:
        out = subprocess.run(
            ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "f32le", "pipe:1"],
            capture_output=True, check=True,
        ).stdout
    except subprocess.CalledProcessError as e:
        raise HTTPException(status_code=400, detail=f"Could not decode this audio: {e.stderr.decode(errors='replace')[:300]}")
    finally:
        os.unlink(path)
    return np.frombuffer(out, dtype=np.float32)

def split_audio(samples: np.ndarray) -> list:
    """Plain, non-overlapping pieces of PIECE_SECONDS. Whisper hears 30 s at a time, and the pipeline's own
    chunked mode (chunk_length_s) was measured dropping speech on 38-54 s clips; plain pieces did not.
    A leftover under 2 s is folded into the previous piece rather than sent alone."""
    step = PIECE_SECONDS * SAMPLE_RATE
    pieces = [samples[i:i + step] for i in range(0, len(samples), step)]
    if len(pieces) > 1 and len(pieces[-1]) < 2 * SAMPLE_RATE:
        leftover = pieces.pop()
        pieces[-1] = np.concatenate([pieces[-1], leftover])
    return pieces

def check_key(authorization: Optional[str]):
    if authorization != f"Bearer {NATLAS_API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid or missing API key")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    language: Optional[str] = None
    max_tokens: Optional[int] = 512
    # Sampling settings the OpenAtlas gateway sends; defaults match it (tested on real N-ATLaS weights).
    temperature: Optional[float] = 0.1
    repetition_penalty: Optional[float] = 1.12
    chat_template_kwargs: Optional[dict] = None

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest, authorization: Optional[str] = Header(None)):
    check_key(authorization)
    # N-ATLaS's chat template otherwise hard-codes "Today Date: 26 Jul 2024".
    template_kwargs = {"date_string": time.strftime("%d %b %Y"), **(req.chat_template_kwargs or {})}
    prompt = llm_tokenizer.apply_chat_template(
        [m.dict() for m in req.messages], tokenize=False, add_generation_prompt=True, **template_kwargs
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    t0 = time.time()
    with gpu_lock, torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 512,
            do_sample=(req.temperature or 0) > 0,
            temperature=req.temperature if (req.temperature or 0) > 0 else None,
            repetition_penalty=req.repetition_penalty or 1.0,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    latency = time.time() - t0
    text = llm_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return {
        "content": text.strip(),
        "model": "n-atlas-llm",
        "latency_seconds": round(latency, 2),
    }

@app.post("/v1/audio/transcriptions")
def transcribe(
    audio: UploadFile = File(...),
    language: str = Form(...),
    authorization: Optional[str] = Header(None),
):
    # A plain def: FastAPI runs it in a worker thread, so decoding and the model call don't block the server.
    check_key(authorization)
    if language not in asr_pipelines:
        raise HTTPException(status_code=400, detail=f"language must be one of {list(asr_pipelines)}")
    samples = decode_audio(audio.file.read())
    if samples.size == 0:
        raise HTTPException(status_code=400, detail="The audio is empty.")
    pieces = split_audio(samples)
    t0 = time.time()
    with gpu_lock:
        texts = [asr_pipelines[language]({"array": p, "sampling_rate": SAMPLE_RATE})["text"].strip() for p in pieces]
    latency = time.time() - t0
    return {
        "text": " ".join(t for t in texts if t),
        "language": language,
        "model": f"n-atlas-asr-{language}",
        "audio_seconds": round(samples.size / SAMPLE_RATE, 2),
        "pieces": len(pieces),
        "latency_seconds": round(latency, 2),
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "llm": "NCAIR1/N-ATLaS",
        "asr_languages": list(asr_pipelines.keys()),
    }

print("FastAPI app defined.")


## 6. Smoke test — call it in-process before exposing it publicly

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
headers = {"Authorization": f"Bearer {NATLAS_API_KEY}"}

r = client.post("/v1/chat/completions", headers=headers, json={
    "messages": [{"role": "user", "content": "Ṣe o le ṣàlàyé ìdí tí ọ̀run fi jẹ́ búlúù?"}],
    "max_tokens": 120,
})
print("chat() smoke test:", r.status_code)
print(r.json())

assert r.status_code == 200 and r.json().get("content"), "LLM smoke test failed — do not proceed to tunnel"
print("\nLLM is genuinely responding. Safe to go public.")


## 7. Run the server + public tunnel

Prints `NATLAS_BASE_URL` at the end — that's what goes into the OpenAtlas SDK / starter kits' env var.

In [ ]:
import nest_asyncio, uvicorn, threading, subprocess, time, re

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(5)

# cloudflared runs in its own session (a notebook interrupt doesn't reach it) and logs to a file
# (a pipe nobody reads fills up and blocks it). Either one made the tunnel drop with Cloudflare 530.
tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True,
)

public_url = None
for _ in range(60):
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if match:
        public_url = match.group(0)
        break
    if tunnel.poll() is not None:
        break
    time.sleep(1)
print(open("cloudflared.log").read()[-1500:])

if public_url:
    print("\n" + "=" * 60)
    print(f"NATLAS_BASE_URL = {public_url}/v1")
    print(f"NATLAS_API_KEY  = {NATLAS_API_KEY}")
    print("=" * 60)
    print("\nPaste these into the OpenAtlas SDK / site env vars.")
    print("This URL dies when this Colab session ends or goes idle — rerun this notebook to get a new one.")
else:
    print("Could not find the tunnel URL in cloudflared output — check the log above.")


## 7b. Optional: speech output (TTS renderer)

A separate, final-stage renderer: it turns text that N-ATLaS has already written into speech, and does nothing
else (no reasoning, translation or transcription; it has no access to the N-ATLaS models). Engines: **sorotts**
(Orpheus-3B + LoRA; Yoruba, Hausa, Igbo, Pidgin) with **MMS-TTS** as fallback and for English. Source:
`deploy/server/tts_renderer.py` in the OpenAtlas repo.

**Already running the server?** Stop the cell in section 8 (the loop; the server and tunnel keep running),
run the two cells below, then start section 8 again. The tunnel URL does not change.
To leave TTS out, set `ENABLE_TTS = False` (or skip these cells).

In [ ]:
%%writefile tts_renderer.py
"""OpenAtlas speech renderer (optional, separable): turns text into speech, nothing else.

It is the last step of a pipeline, after N-ATLaS has produced the text. It never reasons, translates or
transcribes: it receives only the HTTP app, the API-key check and the GPU lock, and has no access to the
N-ATLaS LLM or ASR models. Remove it by not calling install() (or deleting this file); nothing else
depends on it.

    POST /v1/audio/speech   JSON {"text", "language": en|ha|yo|ig|pcm, "engine": auto|sorotts|mms}
                            -> {"audio_base64" (WAV), "format", "sample_rate", "seconds", "engine", "model",
                                "voice", "sentences", "latency_seconds", "fallback_reason"?, "warnings"}
    GET  /health            the server's own health, plus a "tts" section

Engines:
    sorotts  Shinzmann/sorotts: a LoRA adapter on hypaai/hypaai_orpheus_v5 (Orpheus-3B) + SNAC 24 kHz
             decoder. Yoruba, Hausa, Igbo, Nigerian Pidgin. CC BY-NC-SA 4.0. Slow (autoregressive).
    mms      Meta MMS-TTS (VITS): facebook/mms-tts-{eng,hau,yor,pcm}; for Igbo, Shinzmann/soro-tts-ibo
             (an MMS-TTS fine-tune; facebook/mms-tts-ibo is not publicly available). CC BY-NC 4.0. Fast.
    auto     sorotts for a single sentence where it covers the language and is loaded; mms for longer
             text, for other languages, or if sorotts fails.

Environment (natlas_server.py): ENABLE_TTS=1 turns this on; TTS_SOROTTS=0 loads MMS only;
SOROTTS_QUANT="4bit" (default, fits next to N-ATLaS on a 16 GB T4) or "none".
"""

import base64
import io
import os
import re
import threading
import time
import traceback
import wave
from typing import Optional

import numpy as np
import torch
from fastapi import Header, HTTPException
from pydantic import BaseModel

SOROTTS_ADAPTER = "Shinzmann/sorotts"
SOROTTS_BASE = "hypaai/hypaai_orpheus_v5"
SNAC_REPO = "hubertsiuzdak/snac_24khz"
SOROTTS_VOICES = {"yo": "Yor1", "ha": "Hau1", "ig": "Ibo1", "pcm": "NaijaA"}  # the cleanest voice per language (model card)
SOROTTS_RATE = 24000
MMS_REPOS = {
    "en": "facebook/mms-tts-eng",
    "ha": "facebook/mms-tts-hau",
    "yo": "facebook/mms-tts-yor",
    "ig": "Shinzmann/soro-tts-ibo",
    "pcm": "facebook/mms-tts-pcm",
}
LANGUAGES = sorted(MMS_REPOS)
MAX_CHARS = 1000
MAX_SENTENCE_CHARS = 220
SOROTTS_MAX_NEW_TOKENS = 1200  # ~7 SNAC tokens per 12 ms frame: about 14 s of audio per sentence
SILENCE_SECONDS = 0.25
# Orpheus token scheme (must match training; from the sorotts model card).
SOH, EOT, EOH, SOS, EOS, EOAI, OFF = 128259, 128009, 128260, 128257, 128258, 128262, 128266

tts_state = {"status": "loading", "stage": "starting", "error": None, "sorotts": False, "sorotts_error": None, "mms": [], "load_seconds": {}}
_mms = {}
_so = {}
_device = "cuda" if torch.cuda.is_available() else "cpu"


def _load_sorotts(hf_token):
    from huggingface_hub import snapshot_download
    from peft import PeftModel
    from snac import SNAC
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    # The base repo ships its own adapter; load it without, then attach SoroTTS (as the model card does).
    local = snapshot_download(SOROTTS_BASE, ignore_patterns=["adapter_*", "runs/*", "training_args.bin", "handler.py"], token=hf_token)
    tok = AutoTokenizer.from_pretrained(local)
    kwargs = {"device_map": {"": 0} if _device == "cuda" else None}
    if os.environ.get("SOROTTS_QUANT", "4bit") == "4bit" and _device == "cuda":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
        )
    else:
        kwargs["torch_dtype"] = torch.float16 if _device == "cuda" else torch.float32
    base = AutoModelForCausalLM.from_pretrained(local, **kwargs)
    model = PeftModel.from_pretrained(base, SOROTTS_ADAPTER, token=hf_token).eval()
    snac = SNAC.from_pretrained(SNAC_REPO).to(_device).eval()
    _so.update(tok=tok, model=model, snac=snac)


def load(hf_token=None, load_sorotts=True):
    try:
        from transformers import AutoTokenizer, VitsModel

        for lang, repo in MMS_REPOS.items():
            tts_state["stage"] = f"loading {repo}"
            t0 = time.time()
            _mms[lang] = (VitsModel.from_pretrained(repo, token=hf_token).to(_device).eval(), AutoTokenizer.from_pretrained(repo, token=hf_token))
            tts_state["load_seconds"][repo] = round(time.time() - t0, 1)
            tts_state["mms"] = sorted(_mms)
        if load_sorotts:
            tts_state["stage"] = f"loading {SOROTTS_ADAPTER} on {SOROTTS_BASE}"
            t0 = time.time()
            try:
                _load_sorotts(hf_token)
                tts_state["sorotts"] = True
                tts_state["load_seconds"][SOROTTS_ADAPTER] = round(time.time() - t0, 1)
            except Exception as e:  # MMS still serves every language
                _so.clear()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                tts_state["sorotts_error"] = f"{type(e).__name__}: {e}"
                traceback.print_exc()
        tts_state["status"], tts_state["stage"] = "ok", "ready"
        if torch.cuda.is_available():
            print(f"[openatlas-tts] ready. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB", flush=True)
    except Exception as e:
        tts_state["status"], tts_state["error"] = "error", f"{type(e).__name__}: {e}"
        traceback.print_exc()


def split_sentences(text: str):
    """One sentence per generation (sorotts has a ~15 s budget per call); long ones are cut at a comma or space."""
    out = []
    for part in re.split(r"(?<=[.!?])\s+|\n+", text.strip()):
        part = part.strip()
        while len(part) > MAX_SENTENCE_CHARS:
            cut = part.rfind(",", 0, MAX_SENTENCE_CHARS)
            if cut < 40:
                cut = part.rfind(" ", 0, MAX_SENTENCE_CHARS)
            if cut <= 0:
                cut = MAX_SENTENCE_CHARS
            out.append(part[: cut + 1].strip())
            part = part[cut + 1 :].strip()
        if part:
            out.append(part)
    return out


def _snac_decode(codes):
    clamp = lambda v: 0 if v < 0 else (4095 if v > 4095 else v)  # a stray code must not crash SNAC
    l1, l2, l3 = [], [], []
    for i in range(len(codes) // 7):
        b = 7 * i
        l1.append(clamp(codes[b]))
        l2.append(clamp(codes[b + 1] - 4096))
        l3.append(clamp(codes[b + 2] - 2 * 4096))
        l3.append(clamp(codes[b + 3] - 3 * 4096))
        l2.append(clamp(codes[b + 4] - 4 * 4096))
        l3.append(clamp(codes[b + 5] - 5 * 4096))
        l3.append(clamp(codes[b + 6] - 6 * 4096))
    if not l1:
        return np.zeros(0, dtype=np.float32)
    layers = [torch.tensor(x).unsqueeze(0).to(_device) for x in (l1, l2, l3)]
    with torch.inference_mode():
        return _so["snac"].decode(layers).squeeze().float().cpu().numpy()


def _sorotts_sentence(text: str, voice: str):
    tok, model = _so["tok"], _so["model"]
    ids = tok(f"{voice}: {text}", return_tensors="pt").input_ids
    ids = torch.cat([torch.tensor([[SOH]]), ids, torch.tensor([[EOT, EOH]])], dim=1).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=SOROTTS_MAX_NEW_TOKENS,
            do_sample=True, temperature=0.55, top_p=0.95, repetition_penalty=1.1, eos_token_id=[EOS, EOAI], use_cache=True,
        )[0]
    hit_cap = out.shape[0] - ids.shape[1] >= SOROTTS_MAX_NEW_TOKENS
    sos = (out == SOS).nonzero(as_tuple=True)[0]
    seq = out[sos[-1].item() + 1 :] if len(sos) else out[ids.shape[1] :]
    seq = seq[seq >= OFF]  # keep only SNAC audio codes
    n = (seq.size(0) // 7) * 7
    return _snac_decode([t.item() - OFF for t in seq[:n]]), hit_cap


def _mms_sentence(text: str, language: str):
    model, tok = _mms[language]
    inputs = tok(text, return_tensors="pt").to(_device)
    if inputs["input_ids"].shape[1] == 0:
        return np.zeros(0, dtype=np.float32)
    with torch.inference_mode():
        return model(**inputs).waveform[0].float().cpu().numpy()


def to_wav(samples: np.ndarray, rate: int) -> bytes:
    pcm = (np.clip(samples, -1.0, 1.0) * 32767).astype("<i2")
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(pcm.tobytes())
    return buf.getvalue()


def render(text: str, language: str, engine: str, gpu_lock):
    sentences = split_sentences(text)
    if not sentences:
        raise HTTPException(400, "text is empty")
    use = engine
    if engine == "auto":
        # sorotts renders at ~10 s per second of audio on a T4, so anything past one sentence outlasts a
        # quick tunnel's ~100 s timeout (REPORT.md section 24). auto keeps it for single sentences only.
        use = "sorotts" if tts_state["sorotts"] and language in SOROTTS_VOICES and len(sentences) == 1 else "mms"
    warnings, fallback_reason = [], None
    started = time.time()
    if use == "sorotts":
        voice = SOROTTS_VOICES[language]
        try:
            parts = []
            for s in sentences:
                with gpu_lock:  # released between sentences, so chat/ASR requests can interleave
                    audio, hit_cap = _sorotts_sentence(s, voice)
                if hit_cap:
                    warnings.append(f"sorotts reached its token limit on a sentence ({len(s)} chars); the audio may be cut off or garbled there.")
                if audio.size == 0:
                    raise RuntimeError("sorotts produced no audio codes for a sentence")
                parts.append(audio)
            rate, model, used_voice = SOROTTS_RATE, SOROTTS_ADAPTER, voice
        except Exception as e:
            if engine != "auto":
                raise HTTPException(500, f"sorotts failed: {type(e).__name__}: {e}")
            fallback_reason = f"sorotts failed: {type(e).__name__}: {e}"
            use, warnings = "mms", []
    if use == "mms":
        parts = []
        for s in sentences:
            with gpu_lock:
                audio = _mms_sentence(s, language)
            if audio.size:
                parts.append(audio)
        if not parts:
            raise HTTPException(400, "None of this text could be spoken by MMS-TTS (no characters it knows).")
        rate, model, used_voice = _mms[language][0].config.sampling_rate, MMS_REPOS[language], None
    gap = np.zeros(int(SILENCE_SECONDS * rate), dtype=np.float32)
    joined = np.concatenate([x for p in parts for x in (p, gap)][:-1])
    result = {
        "audio_base64": base64.b64encode(to_wav(joined, rate)).decode(),
        "format": "wav",
        "sample_rate": rate,
        "seconds": round(joined.size / rate, 2),
        "engine": use,
        "model": model,
        "voice": used_voice,
        "sentences": len(sentences),
        "latency_seconds": round(time.time() - started, 2),
        "warnings": warnings,
    }
    if fallback_reason:
        result["fallback_reason"] = fallback_reason
    return result


class SpeechReq(BaseModel):
    text: str
    language: str
    engine: Optional[str] = "auto"


def public_state():
    return {
        "status": tts_state["status"],
        "stage": tts_state["stage"],
        "error": tts_state["error"],
        "engines": (["sorotts"] if tts_state["sorotts"] else []) + (["mms"] if _mms else []),
        "sorotts_error": tts_state["sorotts_error"],
        "languages": LANGUAGES,
        "load_seconds": tts_state["load_seconds"],
    }


def install(app, check_key, gpu_lock, hf_token=None, load_sorotts=True, background=True):
    """Adds POST /v1/audio/speech and a "tts" section in GET /health to an existing FastAPI app, then
    loads the models (in a background thread by default). Safe to call again (e.g. re-running a cell)."""
    from fastapi.routing import APIRoute

    if not hasattr(app.state, "health_without_tts"):
        old = next((r for r in app.router.routes if isinstance(r, APIRoute) and r.path == "/health"), None)
        app.state.health_without_tts = old.endpoint if old else (lambda: {"status": "ok"})
    app.router.routes[:] = [r for r in app.router.routes if not (isinstance(r, APIRoute) and r.path in ("/health", "/v1/audio/speech"))]

    @app.get("/health")
    def health_with_tts():
        return {**app.state.health_without_tts(), "tts": public_state()}

    @app.post("/v1/audio/speech")
    def speech(req: SpeechReq, authorization: Optional[str] = Header(None)):
        # A plain def: FastAPI runs it in a worker thread, so a slow render doesn't block the server.
        check_key(authorization)
        if tts_state["status"] != "ok":
            raise HTTPException(503, f"speech renderer not ready: {tts_state['status']} ({tts_state['error'] or tts_state['stage']})")
        if req.language not in MMS_REPOS:
            raise HTTPException(400, f"language must be one of {LANGUAGES}")
        if req.engine not in ("auto", "sorotts", "mms"):
            raise HTTPException(400, "engine must be auto, sorotts or mms")
        if req.engine == "sorotts" and req.language not in SOROTTS_VOICES:
            raise HTTPException(400, f"sorotts covers {sorted(SOROTTS_VOICES)}; use engine mms or auto for {req.language}")
        if req.engine == "sorotts" and not tts_state["sorotts"]:
            raise HTTPException(503, f"sorotts is not loaded ({tts_state['sorotts_error'] or 'disabled'})")
        text = (req.text or "").strip()
        if not text:
            raise HTTPException(400, "text is empty")
        if len(text) > MAX_CHARS:
            raise HTTPException(400, f"text is limited to {MAX_CHARS} characters")
        return render(text, req.language, req.engine, gpu_lock)

    if background:
        threading.Thread(target=load, args=(hf_token, load_sorotts), daemon=True).start()
    else:
        load(hf_token, load_sorotts)

In [ ]:
ENABLE_TTS = True  # False skips the optional speech renderer entirely; nothing else changes.

if ENABLE_TTS:
    import importlib, subprocess, sys, time, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "snac", "peft"], check=True)
    sys.path.insert(0, ".")
    import tts_renderer
    importlib.reload(tts_renderer)
    # Adds POST /v1/audio/speech and a "tts" section in /health to the server that is already running,
    # then loads the TTS models in the background. The N-ATLaS models are not touched.
    tts_renderer.install(app, check_key, gpu_lock, HF_TOKEN)

    last = None
    while tts_renderer.tts_state["status"] == "loading":
        if tts_renderer.tts_state["stage"] != last:
            last = tts_renderer.tts_state["stage"]
            print(time.strftime("%H:%M:%S"), last)
        time.sleep(5)
    print(tts_renderer.public_state())
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    # In-process check, one sentence per engine (Hausa).
    for engine in ["mms", "sorotts"]:
        if engine == "sorotts" and not tts_renderer.tts_state["sorotts"]:
            print("sorotts not loaded:", tts_renderer.tts_state["sorotts_error"])
            continue
        r = tts_renderer.render("Sannu, mun karɓi saƙonka kuma za mu taimaka maka.", "ha", engine, gpu_lock)
        print(engine, r["model"], f'{r["seconds"]} s of audio in {r["latency_seconds"]} s', r["warnings"])

## 8. Keep this cell running

Leave this notebook open and this cell running for as long as you need the endpoint up for development,
demo recording, or judges to reach it. Closing the tab / letting the runtime idle out kills it.

In [ ]:
print("Server + tunnel are live. Keep this tab open.")
print("Re-run cell 7 if the tunnel drops (Colab networking hiccup) — same NATLAS_API_KEY, new URL.")
while True:
    time.sleep(60)
